In [1]:
import os
import glob

import pandas as pd

import fiftyone as fo

from tator_tools.download_media import MediaDownloader
from tator_tools.fiftyone_clustering import FiftyOneDatasetViewer
from tator_tools.download_query import QueryDownloader
from tator_tools.yolo_dataset import YOLODataset
from tator_tools.yolo_crop_regions import YOLORegionCropper
from tator_tools.train_model import ModelTrainer
from tator_tools.inference_video import VideoInferencer

from yolo_tiler import YoloTiler, TileConfig

In [ ]:
"eyJtZXRob2QiOiJBTkQiLCJvcGVyYXRpb25zIjpbeyJtZXRob2QiOiJPUiIsIm9wZXJhdGlvbnMiOlt7ImF0dHJpYnV0ZSI6IkNvbW1vbk5hbWUiLCJvcGVyYXRpb24iOiJpY29udGFpbnMiLCJpbnZlcnNlIjpmYWxzZSwidmFsdWUiOiJmaXNoIn0seyJhdHRyaWJ1dGUiOiJTY2llbnRpZmljTmFtZSIsIm9wZXJhdGlvbiI6Imljb250YWlucyIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6InBvbmcifSx7ImF0dHJpYnV0ZSI6IlNjaWVudGlmaWNOYW1lIiwib3BlcmF0aW9uIjoiaWNvbnRhaW5zIiwiaW52ZXJzZSI6ZmFsc2UsInZhbHVlIjoidGhlc2VhIn0seyJhdHRyaWJ1dGUiOiJTY2llbnRpZmljTmFtZSIsIm9wZXJhdGlvbiI6Imljb250YWlucyIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6InN3aWZ0aWEifSx7ImF0dHJpYnV0ZSI6IlNjaWVudGlmaWNOYW1lIiwib3BlcmF0aW9uIjoiaWNvbnRhaW5zIiwiaW52ZXJzZSI6ZmFsc2UsInZhbHVlIjoic3RpY2hvcCJ9LHsiYXR0cmlidXRlIjoiU2NpZW50aWZpY05hbWUiLCJvcGVyYXRpb24iOiJpY29udGFpbnMiLCJpbnZlcnNlIjpmYWxzZSwidmFsdWUiOiJwYXJhbXVyaSJ9LHsiYXR0cmlidXRlIjoiU2NpZW50aWZpY05hbWUiLCJvcGVyYXRpb24iOiJpY29udGFpbnMiLCJpbnZlcnNlIjpmYWxzZSwidmFsdWUiOiJtdXJpY2VhIn0seyJhdHRyaWJ1dGUiOiJTY2llbnRpZmljTmFtZSIsIm9wZXJhdGlvbiI6Imljb250YWlucyIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6Im1hZHJlcG9yYSJ9LHsiYXR0cmlidXRlIjoiU2NpZW50aWZpY05hbWUiLCJvcGVyYXRpb24iOiJpY29udGFpbnMiLCJpbnZlcnNlIjpmYWxzZSwidmFsdWUiOiJtYWRyYWNpcyJ9LHsiYXR0cmlidXRlIjoiU2NpZW50aWZpY05hbWUiLCJvcGVyYXRpb24iOiJpY29udGFpbnMiLCJpbnZlcnNlIjpmYWxzZSwidmFsdWUiOiJlbGxpc2VsbGlkIn0seyJhdHRyaWJ1dGUiOiJTY2llbnRpZmljTmFtZSIsIm9wZXJhdGlvbiI6Imljb250YWlucyIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6ImJlYnJ5Y2UifSx7ImF0dHJpYnV0ZSI6IlNjaWVudGlmaWNOYW1lIiwib3BlcmF0aW9uIjoiaWNvbnRhaW5zIiwiaW52ZXJzZSI6ZmFsc2UsInZhbHVlIjoiYW50aXBhdGhlcyBmdXJjYXRhIn0seyJhdHRyaWJ1dGUiOiJTY2llbnRpZmljTmFtZSIsIm9wZXJhdGlvbiI6Imljb250YWlucyIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6ImFudGlwYXRoZXMgYXRsYW50aWNhIn1dfSx7ImF0dHJpYnV0ZSI6IkluZGl2aWR1YWxDb3VudCIsIm9wZXJhdGlvbiI6ImVxIiwiaW52ZXJzZSI6ZmFsc2UsInZhbHVlIjoiMSJ9LHsiYXR0cmlidXRlIjoiJHZlcnNpb24iLCJvcGVyYXRpb24iOiJlcSIsImludmVyc2UiOnRydWUsInZhbHVlIjoxMTkzfSx7ImF0dHJpYnV0ZSI6IiRjcmVhdGVkX2J5Iiwib3BlcmF0aW9uIjoiZXEiLCJpbnZlcnNlIjp0cnVlLCJ2YWx1ZSI6MzA5fSx7ImF0dHJpYnV0ZSI6IiR2ZXJzaW9uIiwib3BlcmF0aW9uIjoiZXEiLCJpbnZlcnNlIjp0cnVlLCJ2YWx1ZSI6MjI4fSx7ImF0dHJpYnV0ZSI6IiR2ZXJzaW9uIiwib3BlcmF0aW9uIjoiZXEiLCJpbnZlcnNlIjp0cnVlLCJ2YWx1ZSI6NTU3fSx7ImF0dHJpYnV0ZSI6IiRjcmVhdGVkX2J5Iiwib3BlcmF0aW9uIjoiZXEiLCJpbnZlcnNlIjp0cnVlLCJ2YWx1ZSI6NDI4fSx7ImF0dHJpYnV0ZSI6IiRjcmVhdGVkX2J5Iiwib3BlcmF0aW9uIjoiZXEiLCJpbnZlcnNlIjp0cnVlLCJ2YWx1ZSI6MzUyfSx7Im1ldGhvZCI6Ik9SIiwib3BlcmF0aW9ucyI6W3siYXR0cmlidXRlIjoiJHR5cGUiLCJvcGVyYXRpb24iOiJlcSIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6MjQ3fV19XX0="

# Download Query from Tator

In [2]:
# Set parameters
api_token = os.getenv("TATOR_TOKEN")
project_id = 70  # 155

# Search string comes from Tator's Data Metadata Export utility
search_string = "eyJtZXRob2QiOiJBTkQiLCJvcGVyYXRpb25zIjpbeyJhdHRyaWJ1dGUiOiIkY3JlYXRlZF9ieSIsIm9wZXJhdGlvbiI6ImVxIiwiaW52ZXJzZSI6dHJ1ZSwidmFsdWUiOjI1MX0seyJtZXRob2QiOiJPUiIsIm9wZXJhdGlvbnMiOlt7ImF0dHJpYnV0ZSI6IlNjaWVudGlmaWNOYW1lIiwib3BlcmF0aW9uIjoiaWNvbnRhaW5zIiwiaW52ZXJzZSI6ZmFsc2UsInZhbHVlIjoidGhlc2VhIn0seyJhdHRyaWJ1dGUiOiJTY2llbnRpZmljTmFtZSIsIm9wZXJhdGlvbiI6Imljb250YWlucyIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6InN3aWZ0aWEifSx7ImF0dHJpYnV0ZSI6IlNjaWVudGlmaWNOYW1lIiwib3BlcmF0aW9uIjoiaWNvbnRhaW5zIiwiaW52ZXJzZSI6ZmFsc2UsInZhbHVlIjoic3RpY2hvcCJ9LHsiYXR0cmlidXRlIjoiU2NpZW50aWZpY05hbWUiLCJvcGVyYXRpb24iOiJpY29udGFpbnMiLCJpbnZlcnNlIjpmYWxzZSwidmFsdWUiOiJwYXJhbXVyaSJ9LHsiYXR0cmlidXRlIjoiU2NpZW50aWZpY05hbWUiLCJvcGVyYXRpb24iOiJpY29udGFpbnMiLCJpbnZlcnNlIjpmYWxzZSwidmFsdWUiOiJtdXJpY2VhIn0seyJhdHRyaWJ1dGUiOiJTY2llbnRpZmljTmFtZSIsIm9wZXJhdGlvbiI6Imljb250YWlucyIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6Im1hZHJlcG9yYSJ9LHsiYXR0cmlidXRlIjoiU2NpZW50aWZpY05hbWUiLCJvcGVyYXRpb24iOiJpY29udGFpbnMiLCJpbnZlcnNlIjpmYWxzZSwidmFsdWUiOiJtYWRyYWNpcyJ9LHsiYXR0cmlidXRlIjoiU2NpZW50aWZpY05hbWUiLCJvcGVyYXRpb24iOiJpY29udGFpbnMiLCJpbnZlcnNlIjpmYWxzZSwidmFsdWUiOiJlbGxpc2VsbGlkIn0seyJhdHRyaWJ1dGUiOiJTY2llbnRpZmljTmFtZSIsIm9wZXJhdGlvbiI6Imljb250YWlucyIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6ImJlYnJ5Y2UifSx7ImF0dHJpYnV0ZSI6IlNjaWVudGlmaWNOYW1lIiwib3BlcmF0aW9uIjoiaWNvbnRhaW5zIiwiaW52ZXJzZSI6ZmFsc2UsInZhbHVlIjoiYW50aXBhdGhlcyBmdXJjYXRhIn0seyJhdHRyaWJ1dGUiOiJTY2llbnRpZmljTmFtZSIsIm9wZXJhdGlvbiI6Imljb250YWlucyIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6ImFudGlwYXRoZXMgYXRsYW50aWNhIn1dfSx7ImF0dHJpYnV0ZSI6IkluZGl2aWR1YWxDb3VudCIsIm9wZXJhdGlvbiI6ImVxIiwiaW52ZXJzZSI6ZmFsc2UsInZhbHVlIjoiMSJ9LHsiYXR0cmlidXRlIjoiJHZlcnNpb24iLCJvcGVyYXRpb24iOiJlcSIsImludmVyc2UiOnRydWUsInZhbHVlIjoxMTkzfSx7ImF0dHJpYnV0ZSI6IiRjcmVhdGVkX2J5Iiwib3BlcmF0aW9uIjoiZXEiLCJpbnZlcnNlIjp0cnVlLCJ2YWx1ZSI6MzA5fSx7ImF0dHJpYnV0ZSI6IiR2ZXJzaW9uIiwib3BlcmF0aW9uIjoiZXEiLCJpbnZlcnNlIjp0cnVlLCJ2YWx1ZSI6MjI4fSx7ImF0dHJpYnV0ZSI6IiR2ZXJzaW9uIiwib3BlcmF0aW9uIjoiZXEiLCJpbnZlcnNlIjp0cnVlLCJ2YWx1ZSI6NTU3fSx7ImF0dHJpYnV0ZSI6IiRjcmVhdGVkX2J5Iiwib3BlcmF0aW9uIjoiZXEiLCJpbnZlcnNlIjp0cnVlLCJ2YWx1ZSI6NDI4fSx7ImF0dHJpYnV0ZSI6IiRjcmVhdGVkX2J5Iiwib3BlcmF0aW9uIjoiZXEiLCJpbnZlcnNlIjp0cnVlLCJ2YWx1ZSI6MzUyfSx7Im1ldGhvZCI6Ik9SIiwib3BlcmF0aW9ucyI6W3siYXR0cmlidXRlIjoiJHR5cGUiLCJvcGVyYXRpb24iOiJlcSIsImludmVyc2UiOmZhbHNlLCJ2YWx1ZSI6MjQ3fV19XX0="

# Demo for downloading labeled data
frac = 0.25

dataset_name = "MDBC_Transects"
output_dir = "E:/JordanP/Click-a-Coral/data/reduced"

label_field = ["ScientificName", "CommonName", "PercentCover", "IndividualCount"]

In [3]:
# Create a downloader for the labeled data
downloader = QueryDownloader(api_token,
                             project_id=project_id,
                             search_string=search_string,
                             frac=frac,
                             output_dir=output_dir,
                             dataset_name=dataset_name,
                             label_field=label_field,
                             download_width=1920)

NOTE: Authentication successful for jordan.pierce
NOTE: Search string saved to E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\search_string.txt


In [ ]:
# Download the labeled data
downloader.download_data()

In [ ]:
downloader.display_sample(label_column="ScientificName")

In [ ]:
df = downloader.as_dataframe()  # .as_dict()

# Drop any rows without bounding boxes
df = df.dropna(subset=["x", "y", "width", "height"])

# Convert the ScientificNames
df.loc[df['ScientificName'].str.contains("atlantica", case=False, na=False), 'ScientificName'] = "ANTIPATHES_ATLANTICA"
df.loc[df['ScientificName'].str.contains("furcata", case=False, na=False), 'ScientificName'] = "ANTIPATHES_FURCATA"
df.loc[df['ScientificName'].str.contains("bebryce", case=False, na=False), 'ScientificName'] = "BEBRYCE"
df.loc[df['ScientificName'].str.contains("ellisellid", case=False, na=False), 'ScientificName'] = "ELLISELLIDAE"
df.loc[df['ScientificName'].str.contains("madracis", case=False, na=False), 'ScientificName'] = "MADRACIS"
df.loc[df['ScientificName'].str.contains("madrepora", case=False, na=False), 'ScientificName'] = "MADREPORA"
df.loc[df['ScientificName'].str.contains("muricea", case=False, na=False), 'ScientificName'] = "MURICEA_PENDULA"
df.loc[df['ScientificName'].str.contains("paramuri", case=False, na=False), 'ScientificName'] = "PARAMURICEA"
df.loc[df['ScientificName'].str.contains("stichopathes", case=False, na=False), 'ScientificName'] = "STICHOPATHES"
df.loc[df['ScientificName'].str.contains("swiftia", case=False, na=False), 'ScientificName'] = "SWIFTIA_EXERTA"
df.loc[df['ScientificName'].str.contains("thesea", case=False, na=False), 'ScientificName'] = "THESEA_NIVEA"


# Update the label field to be the ScientificName
df['label'] = df['ScientificName']

print(df.shape, df.columns)

df.sample(10) 

In [ ]:
df['label'].value_counts().plot(kind='bar', figsize=(10, 5))

# Convert Data into YOLO-formatted Dataset

In [4]:
# Set parameters
output_dir = "E:/JordanP/Click-a-Coral/data/reduced/MDBC_Transects"
dataset_name = "YOLO_Detection_Dataset"

train_ratio = 0.8
test_ratio = 0.1

task = 'detect' # 'detect' or 'segment'

In [5]:
df = pd.read_csv(os.path.join(output_dir, "filtered_data.csv"))

In [6]:
# Create and process dataset
dataset = YOLODataset(
    data=df,
    output_dir=output_dir,
    dataset_name=dataset_name,
    train_ratio=train_ratio,
    test_ratio=test_ratio,
    task=task,
    format_class_names=True, 
)

In [7]:
# Process the dataset
dataset.process_dataset(move_images=False)  # Makes a copy of the images instead of moving them

Processing YOLO dataset with 8492 annotations...
Dataset split: 4615 train, 578 valid, 576 test images


Writing detection labels:   0%|          | 0/5769 [00:00<?, ?it/s]

Copying images:   0%|          | 0/5769 [00:00<?, ?it/s]

Dataset created at E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset
Classes: ['ANTIPATHES_ATLANTICA', 'ANTIPATHES_FURCATA', 'BEBRYCE', 'ELLISELLIDAE', 'MADRACIS', 'MADREPORA', 'MURICEA_PENDULA', 'STICHOPATHES', 'SWIFTIA_EXERTA', 'THESEA_NIVEA']


Rendering Examples:   0%|          | 0/10 [00:00<?, ?it/s]

Rendered 10 examples to E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\examples


In [8]:
dataset.dataset_dir

'E:\\JordanP\\Click-a-Coral\\data\\reduced\\MDBC_Transects\\YOLO_Detection_Dataset'

In [9]:
os.path.exists(f"{dataset.dataset_dir}\\data.yaml")

True

# Crop Regions (Optional)

In [10]:
cropper = YOLORegionCropper(dataset_path=f"{dataset.dataset_dir}\\data.yaml", 
                            output_dir=f"{os.path.dirname(dataset.dataset_dir)}",
                            dataset_name="YOLO_Classification_Dataset",
                            format_class_names=False)

In [11]:
# Process the dataset to create classification crops
cropper.process_dataset()

NOTE: Loading dataset...
NOTE: Converting dataset...
Added 4615 images from train dataset
Added 578 images from validation dataset
Added 575 images from test dataset


SupervisionWarnings: Passing a `Dict[str, np.ndarray]` into `DetectionDataset` is deprecated and will be removed in `supervision-0.26.0`. Use a list of paths `List[str]` instead.


NOTE: Loaded dataset - 8491 detections found


Creating crops: 100%|██████████| 5768/5768 [00:33<00:00, 173.43it/s]


Created 8490 crops from 5768 images
NOTE: Writing classification dataset YAML...
Classification dataset YAML written to E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects/YOLO_Classification_Dataset/data.yaml
Class distribution:
  ANTIPATHES_ATLANTICA: train=1100, val=342, test=111
  ANTIPATHES_FURCATA: train=1013, val=291, test=146
  BEBRYCE: train=61, val=13, test=6
  ELLISELLIDAE: train=8, val=2, test=2
  MADRACIS: train=113, val=29, test=13
  MADREPORA: train=526, val=147, test=80
  MURICEA_PENDULA: train=233, val=69, test=28
  STICHOPATHES: train=1210, val=396, test=185
  SWIFTIA_EXERTA: train=506, val=135, test=61
  THESEA_NIVEA: train=1141, val=345, test=178
NOTE: Created classification dataset in E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects/YOLO_Classification_Dataset


# Train a YOLO Model

In [ ]:
import torch
from ultralytics.utils.tal import TaskAlignedAssigner

from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils import RANK

# ---------------------------------------------------------------------------
# 1. Custom Assigner (This class is correct and unchanged)
# ---------------------------------------------------------------------------
import torch
import torch.nn as nn
from ultralytics.utils import LOGGER, RANK
from ultralytics.utils.metrics import bbox_iou

# Note: This is now a standalone module, not inheriting from the ultralytics one.
class CustomAssigner(nn.Module):
    """
    A custom task-aligned assigner that is a self-contained copy of the original,
    with an added feature to randomly drop a percentage of negative samples and mask out negatives by score threshold.
    """

    def __init__(self, topk: int = 10, num_classes: int = 80, alpha: float = 0.5, beta: float = 6.0, eps: float = 1e-9, 
                 drop_rate: float = 0.5, score_min_thresh: float = 0, score_max_thresh: float = 1.0):
        """
        Initialize the CustomAssigner.
        Adds `drop_rate`, `score_min_thresh`, and `score_max_thresh` to the original TaskAlignedAssigner's parameters.
        """
        super().__init__()
        self.topk = topk
        self.num_classes = num_classes
        self.alpha = alpha
        self.beta = beta
        self.eps = eps
        self.drop_rate = drop_rate # Our custom parameter
        self.score_min_thresh = score_min_thresh
        self.score_max_thresh = score_max_thresh
        
        # Statistics tracking
        self.register_buffer('total_negatives', torch.tensor(0))
        self.register_buffer('dropped_negatives', torch.tensor(0))
        
        if RANK in (-1, 0):
            print(f"✅ CustomAssigner initialized with negative drop_rate: {self.drop_rate}")

    @torch.no_grad()
    def forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        """
        Computes the assignment and then applies our custom negative dropping logic.
        """
        self.bs = pd_scores.shape[0]
        self.n_max_boxes = gt_bboxes.shape[1]
        device = gt_bboxes.device

        if self.n_max_boxes == 0:
            # Handle case with no ground truths
            return (
                torch.full_like(pd_scores[..., 0], self.num_classes),
                torch.zeros_like(pd_bboxes),
                torch.zeros_like(pd_scores),
                torch.zeros_like(pd_scores[..., 0]),
                torch.zeros_like(pd_scores[..., 0]),
            )

        try:
            # Perform the standard assignment logic by calling the internal _forward method
            results = self._forward(pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)
        except torch.cuda.OutOfMemoryError:
            # Handle OOM by moving to CPU
            LOGGER.warning("CUDA OutOfMemoryError in CustomAssigner, using CPU")
            cpu_tensors = [t.cpu() for t in (pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)]
            results = self._forward(*cpu_tensors)
            results = tuple(t.to(device) for t in results)

        # --- OUR CUSTOM LOGIC STARTS HERE ---
        target_labels, target_bboxes, target_scores, fg_mask, target_gt_idx = results

        if self.training:
            bg_mask = ~fg_mask
            # --- Threshold masking ---
            if self.score_min_thresh is not None or self.score_max_thresh is not None:
                # Get background indices
                bg_indices = torch.where(bg_mask)
                # Gather the prediction scores for background anchors
                bg_scores = pd_scores[bg_indices]
                mask = torch.ones_like(bg_scores, dtype=torch.bool)
                if self.score_min_thresh is not None:
                    mask &= (bg_scores >= self.score_min_thresh)
                if self.score_max_thresh is not None:
                    mask &= (bg_scores <= self.score_max_thresh)
                    
                # Invert mask: True where we want to mask out (set to -1)
                mask_to_drop = ~mask
                # Set target_scores to -1 for masked-out negatives
                target_scores[bg_indices[0][mask_to_drop], bg_indices[1][mask_to_drop]] = -1.0

            # --- Random negative dropping ---
            if self.drop_rate > 0:
                # Identify background samples that are still valid (not positive and not already dropped)
                # A target score of 0 indicates a valid background sample.
                bg_mask = ~fg_mask
                bg_indices = torch.where(bg_mask)
                
                num_neg = len(bg_indices[0])
                num_to_drop = int(num_neg * self.drop_rate)
                self.total_negatives += num_neg
                self.dropped_negatives += num_to_drop
                
                if num_to_drop > 0:
                    perm = torch.randperm(num_neg, device=pd_scores.device)
                    drop_indices_perm = perm[:num_to_drop]
                    batch_idx_to_drop = bg_indices[0][drop_indices_perm]
                    anchor_idx_to_drop = bg_indices[1][drop_indices_perm]
                    target_scores[batch_idx_to_drop, anchor_idx_to_drop] = -1.0

        return target_labels, target_bboxes, target_scores, fg_mask, target_gt_idx

    def get_drop_statistics(self):
        """Get statistics about negative sample dropping."""
        if self.total_negatives > 0:
            drop_rate = (self.dropped_negatives.float() / self.total_negatives.float()).item()
            return {
                'total_negatives': self.total_negatives.item(),
                'dropped_negatives': self.dropped_negatives.item(),
                'actual_drop_rate': drop_rate
            }
        return {'total_negatives': 0, 'dropped_negatives': 0, 'actual_drop_rate': 0.0}

    def reset_statistics(self):
        """Reset drop statistics."""
        self.total_negatives.zero_()
        self.dropped_negatives.zero_()

    def print_drop_statistics(self):
        """Print current drop statistics."""
        stats = self.get_drop_statistics()
        if RANK in (-1, 0):
            print(f"📊 Negative Drop Stats - Total: {stats['total_negatives']}, "
                  f"Dropped: {stats['dropped_negatives']}, "
                  f"Rate: {stats['actual_drop_rate']:.3f}")

    # --- ALL HELPER METHODS ARE COPIED DIRECTLY FROM THE ORIGINAL ---

    def _forward(self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt):
        """Internal forward pass for assignment."""
        mask_pos, align_metric, overlaps = self.get_pos_mask(
            pd_scores, pd_bboxes, gt_labels, gt_bboxes, anc_points, mask_gt
        )
        target_gt_idx, fg_mask, mask_pos = self.select_highest_overlaps(mask_pos, overlaps, self.n_max_boxes)
        target_labels, target_bboxes, target_scores = self.get_targets(gt_labels, gt_bboxes, target_gt_idx, fg_mask)
        align_metric *= mask_pos
        pos_align_metrics = align_metric.amax(dim=-1, keepdim=True)
        pos_overlaps = (overlaps * mask_pos).amax(dim=-1, keepdim=True)
        norm_align_metric = (align_metric * pos_overlaps / (pos_align_metrics + self.eps)).amax(-2).unsqueeze(-1)
        target_scores = target_scores * norm_align_metric
        return target_labels, target_bboxes, target_scores, fg_mask.bool(), target_gt_idx

    def get_pos_mask(self, pd_scores, pd_bboxes, gt_labels, gt_bboxes, anc_points, mask_gt):
        """Get positive mask."""
        mask_in_gts = self.select_candidates_in_gts(anc_points, gt_bboxes)
        align_metric, overlaps = self.get_box_metrics(pd_scores, pd_bboxes, gt_labels, gt_bboxes, mask_in_gts * mask_gt)
        mask_topk = self.select_topk_candidates(align_metric, topk_mask=mask_gt.expand(-1, -1, self.topk).bool())
        mask_pos = mask_topk * mask_in_gts * mask_gt
        return mask_pos, align_metric, overlaps

    def get_box_metrics(self, pd_scores, pd_bboxes, gt_labels, gt_bboxes, mask_gt):
        """Compute alignment metric."""
        na = pd_bboxes.shape[-2]
        mask_gt = mask_gt.bool()
        overlaps = torch.zeros([self.bs, self.n_max_boxes, na], dtype=pd_bboxes.dtype, device=pd_bboxes.device)
        bbox_scores = torch.zeros([self.bs, self.n_max_boxes, na], dtype=pd_scores.dtype, device=pd_scores.device)
        ind = torch.zeros([2, self.bs, self.n_max_boxes], dtype=torch.long)
        ind[0] = torch.arange(end=self.bs).view(-1, 1).expand(-1, self.n_max_boxes)
        ind[1] = gt_labels.squeeze(-1)
        bbox_scores[mask_gt] = pd_scores[ind[0], :, ind[1]][mask_gt]
        pd_boxes = pd_bboxes.unsqueeze(1).expand(-1, self.n_max_boxes, -1, -1)[mask_gt]
        gt_boxes = gt_bboxes.unsqueeze(2).expand(-1, -1, na, -1)[mask_gt]
        overlaps[mask_gt] = self.iou_calculation(gt_boxes, pd_boxes)
        align_metric = bbox_scores.pow(self.alpha) * overlaps.pow(self.beta)
        return align_metric, overlaps

    def iou_calculation(self, gt_bboxes, pd_bboxes):
        """Calculate IoU."""
        return bbox_iou(gt_bboxes, pd_bboxes, xywh=False, CIoU=True).squeeze(-1).clamp_(0)

    def select_topk_candidates(self, metrics, topk_mask=None):
        """Select top-k candidates."""
        topk_metrics, topk_idxs = torch.topk(metrics, self.topk, dim=-1, largest=True)
        if topk_mask is None:
            topk_mask = (topk_metrics.max(-1, keepdim=True)[0] > self.eps).expand_as(topk_idxs)
        topk_idxs.masked_fill_(~topk_mask, 0)
        count_tensor = torch.zeros(metrics.shape, dtype=torch.int8, device=topk_idxs.device)
        ones = torch.ones_like(topk_idxs[:, :, :1], dtype=torch.int8, device=topk_idxs.device)
        for k in range(self.topk):
            count_tensor.scatter_add_(-1, topk_idxs[:, :, k : k + 1], ones)
        count_tensor.masked_fill_(count_tensor > 1, 0)
        return count_tensor.to(metrics.dtype)

    def get_targets(self, gt_labels, gt_bboxes, target_gt_idx, fg_mask):
        """Compute target labels, bboxes and scores."""
        batch_ind = torch.arange(end=self.bs, dtype=torch.int64, device=gt_labels.device)[..., None]
        target_gt_idx = target_gt_idx + batch_ind * self.n_max_boxes
        target_labels = gt_labels.long().flatten()[target_gt_idx]
        target_bboxes = gt_bboxes.view(-1, gt_bboxes.shape[-1])[target_gt_idx]
        target_labels.clamp_(0)
        target_scores = torch.zeros((target_labels.shape[0], target_labels.shape[1], self.num_classes),
                                    dtype=torch.int64, device=target_labels.device)
        target_scores.scatter_(2, target_labels.unsqueeze(-1), 1)
        fg_scores_mask = fg_mask[:, :, None].repeat(1, 1, self.num_classes)
        target_scores = torch.where(fg_scores_mask > 0, target_scores, 0)
        return target_labels, target_bboxes, target_scores

    @staticmethod
    def select_candidates_in_gts(xy_centers, gt_bboxes, eps=1e-9):
        """Select anchors in ground truth boxes."""
        n_anchors = xy_centers.shape[0]
        bs, n_boxes, _ = gt_bboxes.shape
        lt, rb = gt_bboxes.view(-1, 1, 4).chunk(2, 2)
        bbox_deltas = torch.cat((xy_centers[None] - lt, rb - xy_centers[None]), dim=2).view(bs, n_boxes, n_anchors, -1)
        return bbox_deltas.amin(3).gt_(eps)

    @staticmethod
    def select_highest_overlaps(mask_pos, overlaps, n_max_boxes):
        """Handle multiple gt assignments."""
        fg_mask = mask_pos.sum(-2)
        if fg_mask.max() > 1:
            mask_multi_gts = (fg_mask.unsqueeze(1) > 1).expand(-1, n_max_boxes, -1)
            max_overlaps_idx = overlaps.argmax(1)
            is_max_overlaps = torch.zeros(mask_pos.shape, dtype=mask_pos.dtype, device=mask_pos.device)
            is_max_overlaps.scatter_(1, max_overlaps_idx.unsqueeze(1), 1)
            mask_pos = torch.where(mask_multi_gts, is_max_overlaps, mask_pos).float()
            fg_mask = mask_pos.sum(-2)
        target_gt_idx = mask_pos.argmax(-2)
        return target_gt_idx, fg_mask, mask_pos
    
# ---------------------------------------------------------------------------
# 2. Custom Loss Class (A clean way to package the change)
# ---------------------------------------------------------------------------
from typing import Any, Dict, Tuple
from ultralytics.utils.tal import make_anchors


class CustomV8DetectionLoss(v8DetectionLoss):
    """
    A custom version of v8DetectionLoss that is updated to match the newer
    ultralytics structure and is robust to the custom assigner's negative dropping and threshold masking.
    """
    def __init__(self, model):
        super().__init__(model)
        # We need to explicitly define bbox_decode if it's not in the parent
        if not hasattr(self, 'bbox_decode'):
            self.bbox_decode = self.decode_bboxes
        
        drop_rate = getattr(model.args, 'nrd_drop_rate', 0.0)  # <-------------------------- Set the drop rate
        # Replace the default assigner with our self-contained custom one
        self.assigner = CustomAssigner(topk=10, num_classes=self.nc,
                                        alpha=0.5, beta=6.0, drop_rate=drop_rate)
        if RANK in (-1, 0):
            print("✅ CustomV8DetectionLoss initialized and assigner replaced.")

    def __call__(self, preds, batch):
        loss = torch.zeros(3, device=self.device)  # box, cls, dfl
        feats = preds[1] if isinstance(preds, tuple) else preds
        pred_distri, pred_scores = torch.cat([xi.view(feats[0].shape[0], self.no, -1) for xi in feats], 2).split(
            (self.reg_max * 4, self.nc), 1)

        pred_scores = pred_scores.permute(0, 2, 1).contiguous()
        pred_distri = pred_distri.permute(0, 2, 1).contiguous()

        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        imgsz = torch.tensor(feats[0].shape[2:], device=self.device, dtype=dtype) * self.stride[0]
        anchor_points, stride_tensor = make_anchors(feats, self.stride, 0.5)

        targets = torch.cat((batch['batch_idx'].view(-1, 1), batch['cls'].view(-1, 1), batch['bboxes']), 1)
        targets = self.preprocess(targets.to(self.device), batch_size, scale_tensor=imgsz[[1, 0, 1, 0]])
        gt_labels, gt_bboxes = targets.split((1, 4), 2)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0)

        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)

        _, target_bboxes, target_scores, fg_mask, _ = self.assigner(
            pred_scores.detach().sigmoid(),
            (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor,
            gt_labels,
            gt_bboxes,
            mask_gt)
        
        # --- FINAL FIX FOR ALL LOSSES ---

        # Normalization for box/dfl loss (this part is correct)
        target_scores_sum = max(target_scores[target_scores > 0].sum(), 1)

        # Classification loss
        # Create a mask to select only targets that are NOT -1
        cls_mask = (target_scores != -1.0)
        
        # Compute BCE loss only on the valid samples (positives and non-dropped negatives)
        loss_cls_unnormalized = self.bce(pred_scores[cls_mask], target_scores[cls_mask].to(dtype))

        # Normalize the classification loss by the number of positive samples
        num_pos = fg_mask.sum()
        if num_pos > 0:
            loss[1] = loss_cls_unnormalized.sum() / num_pos
        # If no positive samples, cls_loss is 0
        
        # Bbox loss (this part is correct)
        if fg_mask.sum():
            target_bboxes /= stride_tensor
            try:
                loss[0], loss[2] = self.bbox_loss(
                    pred_distri, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask)
            except TypeError:
                loss[0], loss[2] = self.bbox_loss(
                    pred_distri, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum)

        loss[0] *= self.hyp.box
        loss[1] *= self.hyp.cls
        loss[2] *= self.hyp.dfl

        return loss.sum() * batch_size, loss.detach()

# ---------------------------------------------------------------------------
# 3. Corrected Custom Trainer (This is the final, working version)
# ---------------------------------------------------------------------------
class CustomTrainer(DetectionTrainer):
    """
    This trainer injects a custom loss function and registers a callback
    to print and reset the custom assigner's statistics at the end of each epoch.
    """

    def _setup_train(self, world_size):
        """
        Overrides the training setup to:
        1. Replace the model's loss function.
        2. Register our custom end-of-epoch callback for statistics.
        """
        # First, run the standard setup.
        super()._setup_train(world_size)

        # Now, replace the criterion with our custom loss.
        if RANK in (-1, 0):
            print("✅ Replacing the model's criterion with our custom loss function.")
        self.model.criterion = CustomV8DetectionLoss(self.model)

        # Ensure the assigner is on the correct device and in training mode.
        self.model.criterion.assigner.to(self.device)
        self.model.criterion.assigner.train()

        # --- NEW: REGISTER THE CALLBACK ---
        # We add our custom method to be called when the 'on_train_epoch_end' event fires.
        self.add_callback("on_train_epoch_end", self._print_and_reset_stats)
        if RANK in (-1, 0):
            print("✅ Registered custom callback for end-of-epoch statistics.")


    def _print_and_reset_stats(self, trainer):
        """
        This is our custom callback function. It will be called automatically
        at the end of each epoch. The 'trainer' argument is the trainer instance,
        which is passed by the `run_callbacks` method.
        """
        # Access the assigner through the model's criterion
        assigner = self.model.criterion.assigner

        # The assigner's print method already checks for RANK, so this is safe
        # to call without an explicit RANK check here.
        assigner.print_drop_statistics()

        # Reset the statistics for the next epoch
        assigner.reset_statistics()
        

In [16]:
data_dir = "E:/JordanP/Click-a-Coral/data/reduced/MDBC_Transects"
dataset = f"{data_dir}/YOLO_Detection_Dataset/data.yaml"
project = f"{data_dir}/results"

In [ ]:
from ultralytics import YOLO

args = dict(
    model="yolov8n.pt",
    data=dataset,
    project=project,
    name="PU_Loss_0_Multiclass_25",
    task='detect',
    epochs=100,
    patience=10,
    half=True,
    imgsz=640,
    single_cls=False,
    plots=True,
    batch=32,
    workers=0 
)

trainer = CustomTrainer(overrides=args)
trainer.train()

Ultralytics 8.3.101  Python-3.10.16 torch-2.6.0+cu118 CUDA:0 (Tesla T4, 16384MiB)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=E:/JordanP/Click-a-Coral/data/reduced/MDBC_Transects/YOLO_Detection_Dataset/data.yaml, epochs=100, time=None, patience=10, batch=32, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=0, project=E:/JordanP/Click-a-Coral/data/reduced/MDBC_Transects/results, name=PU_Loss_0_Multiclass_25, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=True, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed

train: Scanning E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\labels.cache... 4615 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4615/4615 [00:00<?, ?it/s]

train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122544.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122549.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122552.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122566.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122568.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\train\images\6761625_122572.jpg: 1 duplicate labels removed
train: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Data


val: Scanning E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\labels.cache... 578 images, 0 backgrounds, 0 corrupt: 100%|██████████| 578/578 [00:00<?, ?it/s]

val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_108524.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_122540.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_122743.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_122765.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_122809.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\images\6761625_122817.jpg: 1 duplicate labels removed
val: WARNING  E:\JordanP\Click-a-Coral\data\reduced\MDBC_Transects\YOLO_Detection_Dataset\valid\imag

...

📊 Negative Drop Stats - Total: 0, Dropped: 0, Rate: 0.000


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 10/10 [00:33<00:00,  3.30s/it]

                   all        578        823      0.391      0.408      0.325      0.165



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     47/100      5.26G      1.597     0.8137      1.403         94        640:  19%|█▊        | 27/145 [01:14<06:39,  3.39s/it]